# Baseline: IMDB Sentiment Classification

Trains DistilBERT on IMDB dataset and saves predictions to `submission.parquet`.

The model and dataset are pre-cached via `hf_models` / `hf_datasets` task config.

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AdamW,
    get_scheduler,
)
from datasets import load_dataset

## 1. Load Data

Loads `imdb` from HuggingFace datasets (pre-cached at `/hf_cache`).
Alternatively, loads `train.parquet` if available.

In [ ]:
try:
    dataset = load_dataset("stanfordnlp/imdb", split="train")
    testset = load_dataset("stanfordnlp/imdb", split="test")
    texts = dataset["text"]
    labels = dataset["label"]
    test_texts = testset["text"]
    test_labels = testset["label"]
    print(f"Loaded IMDB from HF cache: {len(texts)} train, {len(test_texts)} test")
except Exception as e:
    print(f"HF dataset not cached, falling back to train.parquet: {e}")
    df = pd.read_parquet("train.parquet")
    texts = df["text"].tolist()
    labels = df["label"].tolist()
    test_texts = texts
    test_labels = labels

num_labels = len(set(labels))
print(f"Classes: {num_labels}")

## 2. Tokenize

In [ ]:
model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = SentimentDataset(texts, labels, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataset = SentimentDataset(test_texts, test_labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## 3. Train

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 1
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}/{num_epochs}  Loss: {total_loss / len(train_loader):.4f}")

## 4. Generate Predictions

Predicts on the test set and saves `submission.parquet`. The eval engine compares against `labels.parquet`.

In [ ]:
model.eval()
all_preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        all_preds.extend(preds.tolist())

df_sub = pd.DataFrame({
    "id": np.arange(len(all_preds)),
    "label": all_preds,
})
df_sub.to_parquet("submission.parquet", index=False)

print(f"Saved submission.parquet with {len(df_sub)} rows")
print(df_sub.head())